This is a notebook that accompanies the [Tavily Certification course](https://app.tavily.com/certification)

## Unit 0 - Setup

In [19]:
# ruff: noqa: E402
# pyright: reportMissingTypeStubs=false
# pyright: reportUnknownVariableType=false
# pyright: reportUnknownMemberType=false
# pyright: reportUnknownArgumentType=false

import json
import sys
from pathlib import Path
from typing import cast

import requests
from tavily import TavilyClient

# Notebook cwd is often backend/notebooks/; put backend/ on sys.path for app imports.
backend_root = Path.cwd().resolve()
if backend_root.name == "notebooks":
    backend_root = backend_root.parent
sys.path.insert(0, str(backend_root))

from app.core.config import config
from app.job_discovery.url_filters import load_url_filters

tavily_client = TavilyClient(api_key=config.job_discovery.tavily_api_key)
filters = load_url_filters()


## Unit 1: Tavily Search Basics

### How to Perform a Basic Search


In [7]:
response = cast(
    dict[str, object],
    tavily_client.search("Who is Leo Messi?"),  # pyright: ignore[reportUnknownMemberType]
)
print(response)


{'query': 'Who is Leo Messi?', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://en.wikipedia.org/wiki/Lionel_Messi', 'title': 'Lionel Messi - Wikipedia', 'content': 'Lionel Andrés "Leo" Messi (born 24 June 1987) is an Argentine professional footballer who plays as a forward "Forward (association football)") for and captains "Captain (association football)") both Major League Soccer (MLS) club Inter Miami and the Argentina national team. Widely regarded as one of the greatest players in history, Messi has set numerous records for individual accolades won throughout his professional footballing career, including eight Ballons d\'Or, six European Golden Shoes, [...] Argentine footballer (born 1987)\n\n"Messi" redirects here. For other uses, see Messi (disambiguation) "Messi (disambiguation)").\n\nLionel Messi [...] Jorge completed military service in the Argentine Armed Forces. On his father\'s side, Messi is of Italian and Spanish descent, the great

### Passing your Tavily API key

In [8]:
headers = {
    "Authorization": f"Bearer {config.job_discovery.tavily_api_key}",
}

response = requests.post(
    "https://api.tavily.com/search",
    json={"query": "Who is Leo Messi?"},
    headers=headers,
)

print(json.dumps(response.json(), indent=2))


{
  "query": "Who is Leo Messi?",
  "follow_up_questions": null,
  "answer": null,
  "images": [],
  "results": [
    {
      "url": "https://en.wikipedia.org/wiki/Lionel_Messi",
      "title": "Lionel Messi - Wikipedia",
      "content": "Lionel Andr\u00e9s \"Leo\" Messi (born 24 June 1987) is an Argentine professional footballer who plays as a forward \"Forward (association football)\") for and captains \"Captain (association football)\") both Major League Soccer (MLS) club Inter Miami and the Argentina national team. Widely regarded as one of the greatest players in history, Messi has set numerous records for individual accolades won throughout his professional footballing career, including eight Ballons d'Or, six European Golden Shoes, [...] Argentine footballer (born 1987)\n\n\"Messi\" redirects here. For other uses, see Messi (disambiguation) \"Messi (disambiguation)\").\n\nLionel Messi [...] Jorge completed military service in the Argentine Armed Forces. On his father's side, Me

## Unit 2: Search Parameters


### Example Usage

In [14]:
# Run Unit 0 to set up the client.

response = cast(
    dict[str, object],
    tavily_client.search(  # pyright: ignore[reportUnknownMemberType]
        query="latest trends in generative AI 2025",
        search_depth="advanced",
        topic="news",
        time_range="week",  # past week
        max_results=10,
        include_raw_content=True,
        include_images=False,
        include_answer=False,
        exclude_domains=sorted(filters.skip_domains),
    ),
)

results = cast(list[dict[str, object]], response["results"])
for res in results:
    print(res["title"], res["url"])
    print(str(res["content"])[:500], "...")


/Volumes/projectDrive/projects/webDev/ai/aptitude-search/backend/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


A new book looks at how AI is rewiring the newsroom, for better and worse https://www.niemanlab.org/2026/08/a-new-book-looks-at-how-ai-is-rewiring-the-newsroom-for-better-and-worse
GenAI also enables some fundamentally novel storytelling opportunities, such as the use of chatbots to create individualized dialogic news experiences. For example, Time magazine launched the Time AI Agent in November 2025, which integrates language understanding, voice synthesis, translation, and search capabilities. Readers can ask questions, request summaries, generate audio versions of stories, and translate reports — all through a single interface that can, for instance, produce an audio su ...
The next AI race is for data, and China wants more of it https://www.scmp.com/tech/tech-trends/article/3363318/china-faces-new-ai-bottleneck-it-runs-out-chinese-language-training-data
Artificial intelligence

TechTech Trends

# China faces new AI bottleneck as it runs out of Chinese-language training data

The gl

## Unit 3: Best Practices & Cost/Performance Optimization

### Example: Optimized Search Call for a Research Agent

In [ ]:
response = cast(
    dict[str, object],
    tavily_client.search(  # pyright: ignore[reportUnknownMemberType]
        query="2025 trends in web-agent search APIs",
        search_depth="advanced",
        max_results=10,
        include_raw_content=True,
        include_images=False,
        include_answer=False,
        exclude_domains=sorted(filters.skip_domains),
    ),
)

results = cast(list[dict[str, object]], response["results"])
for res in results:
    print(res["title"], res["url"])
    print(str(res["content"])[:200], "...")


## Module 3: Advanced Tavily Tools — Extract, Crawl, Map

### Tavily Extract


#### Example Request + Response (Python SDK)


In [20]:
# Run Unit 0 to set up the client.

response = cast(
    dict[str, object],
    tavily_client.extract(  # pyright: ignore[reportUnknownMemberType]
        urls=["https://en.wikipedia.org/wiki/Artificial_intelligence"],
        extract_depth="basic",  # or "advanced"
    ),
)

# Pretty-print the vendor-shaped payload; truncate raw_content for notebook readability.
preview = dict(response)
raw_results = preview.get("results")
results = (
    cast(list[dict[str, object]], raw_results)
    if isinstance(raw_results, list)
    else []
)
preview_results: list[dict[str, object]] = []
for row in results:
    item = dict(row)
    raw = item.get("raw_content")
    if isinstance(raw, str) and len(raw) > 120:
        item["raw_content"] = raw[:120] + "..."
    preview_results.append(item)
preview["results"] = preview_results
print(json.dumps(preview, indent=2))


{
  "results": [
    {
      "url": "https://en.wikipedia.org/wiki/Artificial_intelligence",
      "title": "Artificial intelligence - Wikipedia",
      "raw_content": "[Jump to content](https://en.wikipedia.org/wiki/Artificial_intelligence#bodyContent)\n\n- [x] Main menu \n\nMain menu\n\nmove ...",
      "images": []
    }
  ],
  "failed_results": [],
  "response_time": 0.02,
  "request_id": "93896e1c-baf1-47ee-af3d-15207082b31e"
}


### Tavily Crawl & Map


#### Example Crawl Request + Typical Use-case


In [ ]:
# Run Unit 0 to set up the client.

response = cast(
    dict[str, object],
    tavily_client.crawl(  # pyright: ignore[reportUnknownMemberType]
        url="https://docs.tavily.com",
        max_depth=3,
        max_breadth=30,
        limit=100,
        select_paths=["/documentation/.*", "/sdk/.*"],
        exclude_paths=["/private/.*", "/admin/.*"],
        allow_external=False,
        extract_depth="advanced",
        include_images=False,
    ),
)

results = cast(list[dict[str, object]], response["results"])
for page in results:
    print(page["url"])
    print(str(page["raw_content"])[:200], "...")
